# BirdCLEF 2026 - Spatial-Audio Q2L Inference Pipeline

This is a high-fidelity, self-contained inference notebook for our **Multi-Modal Spatial-Audio Query2Label (Q2L) Neural Network**. It performs sliding-window audio segment extraction, applies geo-spatial coordinate frequency encodings, runs multi-head cross-attention across species queries, and writes a robust, Kaggle-compliant `submission.csv`.

### Features:
1. **Self-Contained Model Definitions:** Clean, dependency-free PyTorch implementation of the `AudioVisionBackbone`, `SpatialMLP`, `MultiLabelAttentionHead`, and `LogitGatedFusion` modules.
2. **Pantanal Coordinate Alignment:** Soundscape segments are natively mapped to Pantanal coordinates during inference for consistent Spatial MLP routing.
3. **Zero-Failure Dry-Run Fallback:** If the offline competition test soundscapes directory is empty, the notebook falls back to using training soundscape samples so that the notebook compiles successfully during submission gating.

In [ ]:
import os
import gc
import sys
import math
import time
import glob
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import soundfile as sf
from pathlib import Path
import ast

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models

import warnings
warnings.filterwarnings('ignore')

In [ ]:
class Config:
    ROOT_DIR = '/kaggle/input/competitions/birdclef-2026'
        
    TRAIN_CSV = os.path.join(ROOT_DIR, 'train.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')
    SAMPLE_SUB_CSV = os.path.join(ROOT_DIR, 'sample_submission.csv')

    # Point this to your trained run checkpoint in your Kaggle Dataset input
    MODEL_PATH = '/kaggle/input/models/punyakdei/pipeline-clef-proto1/pytorch/default/1/best_model_fold_0.pth'
    
    # Audio Setup
    SR = 32000
    WINDOW_SECONDS = 5.0
    CHUNK_LENGTH = int(SR * WINDOW_SECONDS)  # 160,000 samples
    
    # Mel Spectrogram Setup
    N_MELS = 128
    N_FFT = 1024
    HOP_LENGTH = 512
    FMIN = 50
    FMAX = 16000
    
    # Spatial Constants
    L_FREQ = 6
    PANTANAL_LAT_CENTER = -19.05
    PANTANAL_LON_CENTER = -56.75
    
    # Model Head Dimensions
    BACKBONE_NAME = 'resnet34'
    SPECIES_EMB_DIM = 256
    BACKBONE_OUT_DIM = 512
    ATTENTION_HEADS = 4
    SPATIAL_MLP_DIMS = [36, 128, 64, 234]
    NUM_CLASSES = 234
    
CFG = Config()

print("Loading submission columns and initializing target label mappings...")
sample_sub = pd.read_csv(CFG.SAMPLE_SUB_CSV)
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
CFG.NUM_CLASSES = len(submission_labels)
species_to_idx = {species: idx for idx, species in enumerate(submission_labels)}
print(f"Official submission target labels: {CFG.NUM_CLASSES}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active inference hardware: {device}")

In [ ]:
def lat_lon_to_cartesian(lat: torch.Tensor, lon: torch.Tensor) -> torch.Tensor:
    """
    Maps Lat/Lon to 3D Cartesian Coordinates on a unit sphere.
    """
    lat_rad = torch.deg2rad(lat)
    lon_rad = torch.deg2rad(lon)
    x = torch.cos(lat_rad) * torch.cos(lon_rad)
    y = torch.cos(lat_rad) * torch.sin(lon_rad)
    z = torch.sin(lat_rad)
    return torch.stack([x, y, z], dim=-1)

def get_sinusoidal_positional_encoding(coords: torch.Tensor, L: int = 6) -> torch.Tensor:
    """
    Applies sinusoidal positional encoding to a 3D coordinate vector (x, y, z).
    """
    encodings = []
    for i in range(L):
        freq = (2.0 ** i) * np.pi
        encodings.append(torch.sin(coords * freq))
        encodings.append(torch.cos(coords * freq))
    return torch.cat(encodings, dim=-1)
def normalize_spectrogram(x: torch.Tensor) -> torch.Tensor:
    """
    Normalizes a log-mel spectrogram and duplicates to 3 channels with ImageNet mean/std.
    Matches dataset.py training pipeline exactly.
    """
    min_val = x.min()
    max_val = x.max()
    if max_val - min_val > 1e-5:
        x = (x - min_val) / (max_val - min_val)
    else:
        x = torch.zeros_like(x)
    x = x.squeeze(0)  # Remove batch dim to operate on [N_MELS, TIME]
    x = x.unsqueeze(0).repeat(3, 1, 1)  # Expand to 3 channels: [3, N_MELS, TIME]
    
    # ImageNet standard normalization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(x.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(x.device)
    x = (x - mean) / std
    return x.unsqueeze(0)  # Shape: [1, 3, N_MELS, TIME]


In [ ]:
class AudioVisionBackbone(nn.Module):
    def __init__(self, backbone_name="resnet34", pretrained=False):
        super().__init__()
        if backbone_name == "resnet34":
            self.backbone = models.resnet34(weights=None)
            self.features = nn.Sequential(
                self.backbone.conv1, self.backbone.bn1, self.backbone.relu,
                self.backbone.maxpool, self.backbone.layer1, self.backbone.layer2,
                self.backbone.layer3, self.backbone.layer4
            )
            self.out_channels = 512
        elif backbone_name == "efficientnet_b0":
            self.backbone = models.efficientnet_b0(weights=None)
            self.features = self.backbone.features
            self.out_channels = 1280
        else:
            raise ValueError(f"Backbone {backbone_name} is not supported.")
            
    def forward(self, x):
        return self.features(x)

class SpatialMLP(nn.Module):
    def __init__(self, layer_dims=CFG.SPATIAL_MLP_DIMS, dropout_prob=0.2):
        super().__init__()
        layers = []
        for i in range(len(layer_dims) - 1):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
            if i < len(layer_dims) - 2:
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout_prob))
        self.mlp = nn.Sequential(*layers)
        
    def forward(self, pe_coords, apply_dropout=False):
        return self.mlp(pe_coords)

class MultiLabelAttentionHead(nn.Module):
    def __init__(self, num_classes=CFG.NUM_CLASSES, backbone_dim=CFG.BACKBONE_OUT_DIM, embed_dim=CFG.SPECIES_EMB_DIM, num_heads=CFG.ATTENTION_HEADS):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.species_queries = nn.Parameter(torch.randn(num_classes, embed_dim) * 0.02)
        self.feat_projection = nn.Conv2d(backbone_dim, embed_dim, kernel_size=1)
        self.cross_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.cross_norm = nn.LayerNorm(embed_dim)
        self.self_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.self_norm = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, 1)
        
    def forward(self, features):
        batch_size = features.size(0)
        proj_feats = self.feat_projection(features)
        proj_feats = proj_feats.flatten(2).transpose(1, 2)
        queries = self.species_queries.unsqueeze(0).expand(batch_size, -1, -1)
        
        attn_out, _ = self.cross_attn(query=queries, key=proj_feats, value=proj_feats)
        x = self.cross_norm(queries + attn_out)
        
        self_attn_out, _ = self.self_attn(query=x, key=x, value=x)
        x = self.self_norm(x + self_attn_out)
        return self.fc(x).squeeze(-1)

class LogitGatedFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1, dtype=torch.float32))
    def forward(self, logits_audio, logits_spatial):
        alpha_gated = torch.clamp(self.alpha, min=0.0)
        return logits_audio + alpha_gated * logits_spatial

class MultiModalBirdModel(nn.Module):
    def __init__(self, backbone_name=CFG.BACKBONE_NAME, num_classes=CFG.NUM_CLASSES):
        super().__init__()
        self.backbone = AudioVisionBackbone(backbone_name=backbone_name, pretrained=False)
        self.attention_head = MultiLabelAttentionHead(num_classes=num_classes, backbone_dim=self.backbone.out_channels)
        self.spatial_mlp = SpatialMLP(dropout_prob=0.2)
        self.fusion = LogitGatedFusion()
        
    def forward(self, spectrogram, pe_coords, apply_dropout=False):
        features = self.backbone(spectrogram)
        logits_audio = self.attention_head(features)
        logits_spatial = self.spatial_mlp(pe_coords, apply_dropout=apply_dropout)
        return self.fusion(logits_audio, logits_spatial)

In [ ]:
# 1. Load Trained Weights strictly (Fail-Fast: Crash if path is incorrect!)
print("Instantiating MultiModalBirdModel...")
model = MultiModalBirdModel(backbone_name=CFG.BACKBONE_NAME, num_classes=CFG.NUM_CLASSES).to(device)
print(f"Loading weights strictly from: {CFG.MODEL_PATH}")
model.load_state_dict(torch.load(CFG.MODEL_PATH, map_location=device))
model.eval()
print("✅ Multi-Modal Model loaded successfully.")

In [ ]:
# 2. Fallback Logic for Kaggle Gating Checks
TEST_DIR = os.path.join(CFG.ROOT_DIR, 'test_soundscapes')
test_files = []
if os.path.exists(TEST_DIR):
    test_files = sorted(glob.glob(f'{TEST_DIR}/*.ogg'))

if len(test_files) == 0:
    print('FALLBACK ACTIVE: No test files found in test_soundscapes. Using training soundscapes as dry-run.')
    test_files = sorted(glob.glob(f'{CFG.SOUNDSCAPE_DIR}/*.ogg'))[:5]
    IS_DRY_RUN = True
else:
    print(f'Found {len(test_files)} soundscape files in the test directory.')
    IS_DRY_RUN = False

In [ ]:
# 3. Sliding Window Inference Loop
mel_transform = T.MelSpectrogram(
    sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
    n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
).to(device)

# Pre-calculate the fixed coordinate frequency encoding (soundscapes are mapped to the Pantanal center)
lat_tensor = torch.tensor([CFG.PANTANAL_LAT_CENTER], dtype=torch.float32)
lon_tensor = torch.tensor([CFG.PANTANAL_LON_CENTER], dtype=torch.float32)
cartesian = lat_lon_to_cartesian(lat_tensor, lon_tensor)
pe_coords = get_sinusoidal_positional_encoding(cartesian, L=CFG.L_FREQ).to(device)  # Shape [1, 36]

all_predictions = []
all_row_ids = []

print(f"\n{'='*20} Executing Multi-Modal Inference {'='*20}")
for audio_path in tqdm(test_files, desc="Evaluating Audio Channels"):
    filename = os.path.basename(audio_path).replace('.ogg', '')
    
    try:
        # Use soundfile directly to bypass torchaudio backend deprecations
        data, sample_rate = sf.read(audio_path, dtype='float32')
        # Convert to Mono
        if data.ndim > 1:
            data = data.mean(axis=1)
    except Exception as e:
        print(f"Error reading {audio_path}: {e}")
        continue
        
    total_samples = len(data)
    window_samples = int(CFG.SR * CFG.WINDOW_SECONDS)
    n_segments = math.ceil(total_samples / window_samples)
    
    for seg_idx in range(n_segments):
        start_sample = seg_idx * window_samples
        end_sample = start_sample + window_samples
        end_time_sec = int((seg_idx + 1) * CFG.WINDOW_SECONDS)
        row_id = f"{filename}_{end_time_sec}"
        
        # Extract segment and pad with zero if short
        segment = data[start_sample:end_sample]
        if len(segment) < window_samples:
            pad_len = window_samples - len(segment)
            segment = np.pad(segment, (0, pad_len))
            
        # Convert segment to torch tensor
        seg_tensor = torch.tensor(segment, dtype=torch.float32).unsqueeze(0).to(device)  # [1, 160000]
        
        # Transform
        with torch.no_grad():
            mel_spec = mel_transform(seg_tensor)  # [1, N_MELS, TIME]
            log_mel = torch.log(mel_spec + 1e-6)  # log scaling
            
            # Stack 3 identical channels to match standard torchvision/backbone expected inputs
            image = normalize_spectrogram(log_mel) # Shape: [1, 3, N_MELS, TIME]
            
            # Forward through Model
            output = model(image, pe_coords, apply_dropout=False)  # [1, 234]
            probs = torch.sigmoid(output).squeeze(0).cpu().numpy()
            
        all_row_ids.append(row_id)
        all_predictions.append(probs)

print("Inference completed.")

In [ ]:
# 4. Submission Formatting & Integrity Validation
print("Formatting and validating submission file...")
prediction_df = pd.DataFrame(all_predictions, columns=submission_labels)
submission_df = prediction_df.copy()
submission_df.insert(0, 'row_id', all_row_ids)
submission_df = submission_df[['row_id'] + submission_labels]

# Integrity validation checks
expected_cols = len(submission_labels) + 1
if submission_df.shape[1] != expected_cols:
    raise ValueError(f"Submission has {submission_df.shape[1]} columns, expected {expected_cols}.")
if len(submission_df) != len(all_row_ids):
    raise ValueError(f"Submission has {len(submission_df)} rows, expected {len(all_row_ids)}.")
if submission_df.isnull().values.any():
    raise ValueError("Submission contains missing values!")

# Save submission file
submission_path = 'submission.csv'
submission_df.to_csv(submission_path, index=False)
print(f"\nSubmission successfully saved to: {submission_path}")
print(f"File Shape: {submission_df.shape}")
print("Preview of first 3 rows:")
display(submission_df.head(3))